# Chương 8. Khi dữ liệu không sạch

**Câu hỏi mở đầu:** Một giá trị bất thường là lỗi, thiếu dữ liệu hay chính là điều đáng quan tâm?

Notebook này là tài nguyên đồng hành của chương. Mỗi phần đều đi theo nhịp **câu hỏi → dữ liệu → mã → kết quả → diễn giải → kiểm tra bằng chứng**.

## Mục tiêu

- Tái hiện các ví dụ cốt lõi của chương bằng mã có thể chạy lại.
- Kiểm tra giả định trước khi diễn giải output.
- Kết thúc bằng ít nhất một câu hỏi về điều mà kết quả **chưa** cho biết.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("error", category=FutureWarning)
warnings.filterwarnings("error", category=DeprecationWarning)
ROOT = Path.cwd()
DATA = ROOT / "data"
print("Working root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 20)


## 1. Phát hiện → Chẩn đoán → Quyết định → Xử lý → Kiểm tra → Ghi nhận

In [ ]:
raw = pd.read_csv(DATA / "metromart" / "metromart_sales_raw.csv")
print("raw rows:", len(raw))
print(raw.isna().sum()[raw.isna().sum()>0])

## 2. `customer_id` thiếu: không phải lúc nào cũng cần điền

In [ ]:
missing_customer = raw["customer_id"].isna()
print("missing customer:", int(missing_customer.sum()))
print("rate:", round(missing_customer.mean(),4))

Không có bằng chứng để tạo ra một mã khách hàng giả cho khách vãng lai. Giữ missing có thể phản ánh đúng cơ chế dữ liệu hơn.

## 3. Chuẩn hóa nhãn category có kiểm tra

In [ ]:
clean = raw.copy()
category_map={"beverage":"Beverage","BEV":"Beverage"}
clean["category"] = clean["category"].replace(category_map)
print("Beverage variants before:", raw["category"].isin(["Beverage","beverage","BEV"]).sum())
print("Beverage after:", (clean["category"]=="Beverage").sum())

## 4. Quantity âm: hợp lệ hay lỗi?

In [ ]:
neg = clean.loc[clean["quantity"]<0, ["transaction_type","quantity"]]
print(neg.groupby("transaction_type").size())

Âm trong `return` có thể hợp lệ; âm trong `sale` cần điều tra. Không được xóa mọi giá trị âm bằng một quy tắc duy nhất.

## 5. ThermoFlow: hiếm thống kê và hợp lý vật lý

In [ ]:
tf = pd.read_csv(DATA / "thermoflow" / "thermoflow_runs_raw.csv")
print("negative pressure:", int((tf["pressure_kpa"]<0).sum()))
q99=tf["temperature_out_c"].quantile(0.99)
high=tf.loc[tf["temperature_out_c"]>q99, ["operating_mode","temperature_out_c","sensor_status"]]
print("q99 temperature_out:", q99)
display(high.head())

## 6. Kiểm tra trước và sau

In [ ]:
before_categories = raw["category"].nunique(dropna=True)
after_categories = clean["category"].nunique(dropna=True)
print("category count before/after:", before_categories, after_categories)
print("rows preserved:", len(raw)==len(clean))

### Kiểm tra bằng chứng

Làm sạch là can thiệp vào bằng chứng. Sau mỗi quyết định đáng kể, phải kiểm tra số hàng, missingness, phạm vi và ít nhất một kết quả phân tích nhạy cảm với xử lý.

## Thực hành

Lập Cleaning Decision Log cho `discount_pct > 1`: chưa sửa tự động nếu chưa có metadata đủ mạnh để biết đó là 15% hay 1500%.

---
### Bạn đã sẵn sàng sang chương tiếp theo nếu có thể…

- giải thích output bằng lời;
- chỉ ra ít nhất một giả định;
- nói được kết quả chưa cho phép kết luận điều gì.

**Exit check:** Nếu mã chạy không lỗi nhưng câu trả lời trái với ý nghĩa của dữ liệu, bạn sẽ kiểm tra điều gì trước?